In [4]:
# =========================
# STEP 0 — IMPORTS, PATH, FOLDERS
# =========================
import os, re, json
import numpy as np
import pandas as pd

# For display in Jupyter
try:
    from IPython.display import display
except ImportError:
    def display(x): print(x.head() if hasattr(x, "head") else x)

# >>> Your paths
DATA_PATH  = r"C:\Users\dusik\OneDrive\Desktop\Y2S1\Data\raw\diabetic_data.csv"
OUTPUTS_DIR = r"C:\Users\dusik\OneDrive\Desktop\Y2S1\Results\Outputs"
os.makedirs(OUTPUTS_DIR, exist_ok=True)

print("Outputs will be saved to:", OUTPUTS_DIR)

# Common config
PLACEHOLDERS = ["?", "Unknown/Invalid", "Unknown", "None", "N/A", "NA", "NULL", "Not Available"]
TARGET_COL = "readmitted"  # original target
ID_COLS = [c for c in ["encounter_id", "patient_nbr"] if c]  # keep if present

# Helper: safe display
def shape_note(tag, df): print(f"{tag:<28} -> shape: {df.shape}")

Outputs will be saved to: C:\Users\dusik\OneDrive\Desktop\Y2S1\Results\Outputs


In [6]:
# =========================
# STEP 1 — LOAD & STANDARDIZE (safe for reuse)
# =========================
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Could not find dataset at:\n{DATA_PATH}")

df_raw = pd.read_csv(DATA_PATH)
shape_note("Loaded raw", df_raw)
display(df_raw.head())

# Standardize placeholders to NaN; trim whitespace
df = df_raw.replace(PLACEHOLDERS, np.nan)
for c in df.select_dtypes(include=["object"]).columns:
    df[c] = df[c].astype(str).str.strip()

# Keep IDs/target columns if they exist
ids_present = [c for c in ID_COLS if c in df.columns]
keep_cols = ids_present + ([TARGET_COL] if TARGET_COL in df.columns else [])
shape_note("After standardize", df)

Loaded raw                   -> shape: (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,?,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,?,?,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,?,?,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,?,?,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,?,?,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,?,?,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


After standardize            -> shape: (101766, 50)


In [7]:
# =========================
# STEP 2 — PREPARE CATEGORICALS & SPECIAL MAPPINGS
# =========================
# Identify object (categorical) columns excluding the target
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
if TARGET_COL in cat_cols:
    cat_cols.remove(TARGET_COL)

# 2A) Ordinal mappings for known medical result tiers (if present)
ordinal_maps = {}
if "A1Cresult" in df.columns:
    # Order based on clinical meaning: None < >7 < >8 (dataset uses strings like "None", ">7", ">8", "Norm")
    # Common UCI codes: "None", "Norm", ">7", ">8" (some variants exist)
    a1c_map = {"None": 0, "Norm": 1, ">7": 2, ">8": 3}
    ordinal_maps["A1Cresult"] = a1c_map

if "max_glu_serum" in df.columns:
    # UCI codes: "None", "Norm", ">200", ">300"
    glu_map = {"None": 0, "Norm": 1, ">200": 2, ">300": 3}
    ordinal_maps["max_glu_serum"] = glu_map

for col, mp in ordinal_maps.items():
    if col in df.columns:
        df[col] = df[col].map(mp).astype("Int64")

# 2B) Binary mappings for common yes/no style fields (if present)
# (Add more binary columns if you spot them in your dataset)
binary_maps = {
    "change": {"No": 0, "Ch": 1},
    "diabetesMed": {"No": 0, "Yes": 1},
    "gender": {"Female": 0, "Male": 1}  # adjust if 'Unknown/Other' exists
}
for col, mp in binary_maps.items():
    if col in df.columns and df[col].dtype == "object":
        df[col] = df[col].map(mp)

# Refresh lists after mappings (some became numeric)
cat_cols = [c for c in df.select_dtypes(include=["object"]).columns if c != TARGET_COL]
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

shape_note("Post special mappings", df)
print("Categorical columns (first 20):", cat_cols[:20], "...")


Post special mappings        -> shape: (101766, 50)
Categorical columns (first 20): ['race', 'age', 'weight', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose'] ...


In [8]:
# =========================
# STEP 3 — ENCODING STRATEGY (OHE for ≤20 uniques, Freq-Encode for >20)
# =========================
# Rationale:
# - One-Hot Encode (OHE) for low/medium cardinality to keep interpretability.
# - Frequency Encode for high cardinality to avoid column explosion.
MAX_OHE_CARD = 20

enc_report = []  # to document what we did per column
work = df.copy()

# Separate frame to One-Hot and to Frequency-Encode
ohe_cols, freq_cols = [], []
for col in cat_cols:
    nunq = work[col].nunique(dropna=True)
    if nunq <= MAX_OHE_CARD:
        ohe_cols.append(col)
        enc_report.append({"column": col, "strategy": "one_hot", "unique": int(nunq)})
    else:
        freq_cols.append(col)
        enc_report.append({"column": col, "strategy": "frequency_encode", "unique": int(nunq)})

# 3A) Frequency encoding (replace cat with its value counts proportion)
for col in freq_cols:
    vc = work[col].value_counts(dropna=False)
    freq_map = (vc / vc.sum()).to_dict()
    work[col] = work[col].map(freq_map).astype(float)

# 3B) One-Hot encode OHE columns
if ohe_cols:
    # Keep IDs/target out of encoding
    cols_to_keep = keep_cols + [c for c in work.columns if c not in ohe_cols]
    ohe_frame = pd.get_dummies(work[ohe_cols], prefix=ohe_cols, drop_first=True)
    work = pd.concat([work.drop(columns=ohe_cols), ohe_frame], axis=1)

shape_note("After encoding", work)

# Reorder columns: IDs, target, then rest
ordered_cols = []
ordered_cols += [c for c in ids_present if c in work.columns]
if TARGET_COL in work.columns: ordered_cols += [TARGET_COL]
ordered_cols += [c for c in work.columns if c not in ordered_cols]
work = work[ordered_cols]

After encoding               -> shape: (101766, 111)


In [9]:
# =========================
# STEP 4 — SAVE ENCODED DATA & REPORT
# =========================
encoded_csv = os.path.join(OUTPUTS_DIR, "diabetic_encoded.csv")
work.to_csv(encoded_csv, index=False)
print("Saved encoded dataset to:", encoded_csv)

report_df = pd.DataFrame(enc_report).sort_values(["strategy", "unique", "column"])
report_csv = os.path.join(OUTPUTS_DIR, "encoding_report.csv")
report_df.to_csv(report_csv, index=False)
print("Saved encoding report to:", report_csv)

display(report_df.head(20))
display(work.head())

Saved encoded dataset to: C:\Users\dusik\OneDrive\Desktop\Y2S1\Results\Outputs\diabetic_encoded.csv
Saved encoding report to: C:\Users\dusik\OneDrive\Desktop\Y2S1\Results\Outputs\encoding_report.csv


,column,strategy,unique
4,medical_specialty,frequency_encode,73
5,diag_1,frequency_encode,717
6,diag_2,frequency_encode,749
7,diag_3,frequency_encode,790
24,citoglipton,one_hot,1
23,examide,one_hot,1
13,acetohexamide,one_hot,2
28,glimepiride-pioglitazone,one_hot,2
27,glipizide-metformin,one_hot,2
30,metformin-pioglitazone,one_hot,2


,encounter_id,patient_nbr,readmitted,gender,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,change,diabetesMed,race_Asian,race_Caucasian,race_Hispanic,race_Other,race_nan,age_[10-20),age_[20-30),age_[30-40),age_[40-50),age_[50-60),age_[60-70),age_[70-80),age_[80-90),age_[90-100),weight_[0-25),weight_[100-125),weight_[125-150),weight_[150-175),weight_[175-200),weight_[25-50),weight_[50-75),weight_[75-100),weight_nan,payer_code_CH,payer_code_CM,payer_code_CP,payer_code_DM,payer_code_FR,payer_code_HM,payer_code_MC,payer_code_MD,payer_code_MP,payer_code_OG,payer_code_OT,payer_code_PO,payer_code_SI,payer_code_SP,payer_code_UN,payer_code_WC,payer_code_nan,metformin_No,metformin_Steady,metformin_Up,repaglinide_No,repaglinide_Steady,repaglinide_Up,nateglinide_No,nateglinide_Steady,nateglinide_Up,chlorpropamide_No,chlorpropamide_Steady,chlorpropamide_Up,glimepiride_No,glimepiride_Steady,glimepiride_Up,acetohexamide_Steady,glipizide_No,glipizide_Steady,glipizide_Up,glyburide_No,glyburide_Steady,glyburide_Up,tolbutamide_Steady,pioglitazone_No,pioglitazone_Steady,pioglitazone_Up,rosiglitazone_No,rosiglitazone_Steady,rosiglitazone_Up,acarbose_No,acarbose_Steady,acarbose_Up,miglitol_No,miglitol_Steady,miglitol_Up,troglitazone_Steady,tolazamide_Steady,tolazamide_Up,insulin_No,insulin_Steady,insulin_Up,glyburide-metformin_No,glyburide-metformin_Steady,glyburide-metformin_Up,glipizide-metformin_Steady,glimepiride-pioglitazone_Steady,metformin-rosiglitazone_Steady,metformin-pioglitazone_Steady
0,2278392,8222157,NO,0.0,6,25,1,1,0.001562,41,0,1,0,0,0,0.000934,0.003518,0.013983,1,<NA>,<NA>,0,0,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False,True,False,False,True,False,False,True,False,False,True,False,False,False,True,False,False,True,False,False,False,True,False,False,True,False,False,True,False,False,True,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False
1,149190,55629189,>30,0.0,1,1,7,3,0.490822,59,0,18,0,0,0,0.018562,0.014966,0.000678,9,<NA>,<NA>,1,1,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False,True,False,False,True,False,False,True,False,False,True,False,False,False,True,False,False,True,False,False,False,True,False,False,True,False,False,True,False,False,True,False,False,False,False,False,False,False,True,True,False,False,False,False,False,False
2,64410,86047875,NO,0.0,1,1,7,2,0.490822,11,5,13,2,0,1,0.002801,0.059656,0.000364,6,<NA>,<NA>,0,1,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False,True,False,False,True,False,False,True,False,False,True,False,False,False,False,True,False,True,False,False,False,True,False,False,True,False,False,True,False,False,True,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False
3,500364,82442376,NO,1.0,1,1,7,2,0.490822,44,1,16,0,0,0,0.005061,0.000364,0.023161,7,<NA>,<NA>,1,1,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False,True,False,False,True,False,False,True,False,False,True,False,False,False,True,False,False,True,False